In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

In [ ]:
df = pd.read_csv("../../dataset/Cleaned_Suicide_Detection_DL.csv")
df.columns

In [ ]:
X = df["clean_text"]
y = df["labels"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
class SuicideDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.totlist()
        self.labels = labels.tolist()
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        encoding = tokenizer(
            self.texts[index],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return{
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[index], dtype=torch.long)
        }

In [ ]:
train_dataset = SuicideDataset(X_train, y_train)
test_dataset = SuicideDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(device)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0
    print(f"Epoch {epoch+1}/{epochs}")

    loop = tqdm(train_loader, desc="Training")

    for batch in loop:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            labels = labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loop.set_postfix(loss=loss.item())
    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

- Epoch 1 Loss: 2687.6989
- Epoch 2 Loss: 1609.7105
- Epoch 3 Loss: 1003.7794

In [ ]:
model.eval()
preds = []
true_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(true_labels, preds))
print(classification_report(true_labels, preds))

Accuracy: 0.964764449808198
              precision    recall  f1-score   support

           0       0.96      0.97      0.96     23257
           1       0.96      0.96      0.96     23145

    accuracy                           0.96     46402
   macro avg       0.96      0.96      0.96     46402
weighted avg       0.96      0.96      0.96     46402


In [ ]:
model.save_pretrained("../../models/bert_model/")
tokenizer.save_pretrained("../../models/bert_model/")

In [2]:
model_path = "../../models/bert_model"
tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1793.51it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [5]:
text = "I don't feel like i want to live"

encoding = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

prediction = torch.argmax(logits, dim=1).item()
prediction

1